# 📊 Camada Bronze - Fundos Imobiliários (FII)

## 🎯 Objetivo
Este notebook realiza a **ingestão e consolidação de dados brutos** de 5 FIIs de logística na camada Bronze do projeto, seguindo a arquitetura Medallion.

---

## 📁 Estrutura de Dados

### **Fonte dos Dados**
- **Localização**: `/Workspace/Users/tropehe@outlook.com/projeto-fiis/ingestion/data/`
- **Formato**: CSV (importados via GitHub Actions de APIs públicas)
- **Atualização**: Automatizada via GitHub Actions

### **FIIs Processados**
| Ticker | Nome | Setor |
| --- | --- | --- |
| BTLG11 | BTG Pactual Logística | Logística |
| HGLG11 | CSHG Logística | Logística |
| LVBI11 | VBI Logística | Logística |
| VILG11 | Vinci Logística | Logística |
| XPLG11 | XP Log | Logística |

---

## 🗂️ Tabelas Bronze Criadas

### 1. **workspace.bronze.fii_prices** (Cotações)
- **Registros**: 11.788
- **Período**: ~12 anos (2014-2026)
- **Particionamento**: `ticker`
- **Colunas** (10):
  - **Identificação**: `ticker`, `date`
  - **Preços**: `open`, `high`, `low`, `close`, `adj_close`
  - **Volume**: `volume`
  - **Metadados**: `source_file`, `ingestion_timestamp`

**Transformações aplicadas:**
- ✅ Consolidação de múltiplos CSVs via wildcard (`*_cotacoes.csv`)
- ✅ Extração automática de ticker do nome do arquivo
- ✅ Padronização de colunas para snake_case
- ✅ Adição de metadados de rastreabilidade
- ✅ Validação de nulos e duplicatas (0 encontrados)

---

### 2. **workspace.bronze.fii_dividends** (Dividendos)
- **Registros**: 527
- **Período**: 2013-2026
- **Particionamento**: `ticker`
- **Colunas** (10):
  - **Identificação**: `ticker`, `ex_dividend_date`, `payment_date`
  - **Tipo de evento**: `event_type`, `event_type_description`
  - **Valores**: `value`, `original_value`
  - **Ajuste**: `adjusted` (boolean)
  - **Metadados**: `source_file`, `ingestion_timestamp`

**Transformações aplicadas:**
- ✅ Consolidação de múltiplos CSVs via wildcard (`*_dividendos.csv`)
- ✅ Extração automática de ticker do nome do arquivo
- ✅ Padronização de colunas para snake_case
- ✅ **Remoção de colunas redundantes**: `year`, `month`, `day` (sempre zero), `string_value`, `string_original_value` (duplicadas)
- ✅ Validação de nulos e duplicatas (0 encontrados)

**Distribuição de dividendos por ticker:**
- BTLG11: 122 eventos (2016-2026)
- HGLG11: 122 eventos (2016-2026)
- LVBI11: 92 eventos (2018-2026)
- VILG11: 92 eventos (2018-2026)
- XPLG11: 99 eventos (2018-2026)

---

### 3. **workspace.bronze.ifix** (Índice IFIX)
- **Registros**: 3.868
- **Período**: 15,5 anos (2011-2026)
- **Particionamento**: Nenhum (índice único)
- **Colunas** (4):
  - **Identificação**: `date`
  - **Valor**: `close`
  - **Metadados**: `source_file`, `ingestion_timestamp`

**Transformações aplicadas:**
- ✅ Leitura de arquivo único (`IFIX_completo.csv`)
- ✅ Conversão de formato de data brasileiro (DD/MM/YYYY → date)
- ✅ Padronização de colunas para snake_case
- ✅ Validação de nulos e duplicatas (0 encontrados)

**Estatísticas do IFIX:**
- Mínimo: 981,44 pontos
- Máximo: 3.941,62 pontos
- Média: 2.292,09 pontos

---

## ✅ Validações de Qualidade Aplicadas

Para **todas as tabelas bronze**:
1. ✅ **Nulos em colunas críticas**: 0 registros
2. ✅ **Duplicatas por chave primária**: 0 registros
3. ✅ **Contagem de registros**: validada com SELECT COUNT(*)
4. ✅ **Particionamento**: aplicado e testado
5. ✅ **Metadados de rastreabilidade**: source_file e ingestion_timestamp

---

## 🔄 Próximas Etapas

➡️ **Camada Silver** (notebook `02_silver_fii`):
- Remoção de colunas técnicas (source_file, ingestion_timestamp)
- Cálculo de features derivadas (has_trading)
- Renomeação de colunas para clareza de negócio
- Dados limpos e focados em análise

➡️ **Camada Gold** (a criar):
- Cálculo de retorno total (preço + dividendos)
- Features de ML (momentum, volatilidade, correlações)
- Integração com dados macro (SELIC, IPCA, dólar)
- Dataset final para modelagem

---

## 📝 Notas Técnicas

### **Decisões de Design**
1. **Particionamento por ticker**: Otimiza queries filtradas por FII específico
2. **Preservação de dados brutos**: Bronze mantém todas as colunas originais + metadados
3. **Validação rigorosa**: Garante qualidade antes de propagar para Silver
4. **Formato Delta Lake**: Permite ACID, time travel e schema evolution

### **Padrões de Nomenclatura**
- **Tabelas**: `workspace.[camada].[entidade]`
- **Colunas**: `snake_case`
- **Datas**: tipo `date` (não string)
- **Valores numéricos**: `double` ou `integer`

---

**Última Atualização**: 10/08/2026  
**Autor**: Projeto FII ML Predictor  
**Status**: ✅ Camada Bronze Completa e Validada

In [0]:
# Ler arquivo CSV de cotações do FII BTLG11
file_path = "/Workspace/Users/tropehe@outlook.com/projeto-fiis/ingestion/data/BTLG11_cotacoes.csv"

df_cotacoes = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)

# Exibir o schema detectado
print("=== Schema do arquivo ===")
df_cotacoes.printSchema()

# Exibir as primeiras 5 linhas
print("\n=== Primeiras 5 linhas ===")
display(df_cotacoes.limit(5))

In [0]:
from pyspark.sql.functions import regexp_extract, col
import os

# Caminho base dos arquivos
base_path = "/Workspace/Users/tropehe@outlook.com/projeto-fiis/ingestion/data"

# Ler todos os arquivos *_cotacoes.csv usando wildcard
df_all_cotacoes = spark.read.csv(
    f"{base_path}/*_cotacoes.csv",
    header=True,
    inferSchema=True
)

# Extrair o ticker do nome do arquivo usando _metadata.file_path (Unity Catalog)
df_with_ticker = df_all_cotacoes.withColumn(
    "ticker",
    regexp_extract(col("_metadata.file_path"), r"([A-Z0-9]+)_cotacoes\.csv$", 1)
)

# Exibir schema final
print("=== Schema Final ===")
df_with_ticker.printSchema()

# Contar número total de linhas
total_rows = df_with_ticker.count()
print(f"\n=== Número Total de Linhas: {total_rows:,} ===")

# Contar linhas por ticker
print("\n=== Quantidade de Linhas por Ticker ===")
df_count_by_ticker = df_with_ticker.groupBy("ticker").count().orderBy("ticker")
display(df_count_by_ticker)

# Padronizar nomes das colunas para snake_case
df_final = df_with_ticker \
    .withColumnRenamed("Date", "date") \
    .withColumnRenamed("Adj Close", "adj_close") \
    .withColumnRenamed("Close", "close") \
    .withColumnRenamed("High", "high") \
    .withColumnRenamed("Low", "low") \
    .withColumnRenamed("Open", "open") \
    .withColumnRenamed("Volume", "volume")

# Exibir schema final padronizado
print("\n=== Schema Final Padronizado (snake_case) ===")
df_final.printSchema()

# Exibir 5 registros
print("\n=== 5 Primeiros Registros ===")
display(df_final.limit(5))

In [0]:
from pyspark.sql.functions import regexp_extract, col, current_timestamp, lit
from datetime import datetime

# Caminho base dos arquivos
base_path = "/Workspace/Users/tropehe@outlook.com/projeto-fiis/ingestion/data"

# Ler todos os arquivos *_cotacoes.csv
df_raw = spark.read.csv(
    f"{base_path}/*_cotacoes.csv",
    header=True,
    inferSchema=True
)

# Adicionar metadados de ingestão
df_bronze = df_raw \
    .withColumn("ticker", regexp_extract(col("_metadata.file_path"), r"([A-Z0-9]+)_cotacoes\.csv$", 1)) \
    .withColumn("source_file", col("_metadata.file_path")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumnRenamed("Date", "date") \
    .withColumnRenamed("Adj Close", "adj_close") \
    .withColumnRenamed("Close", "close") \
    .withColumnRenamed("High", "high") \
    .withColumnRenamed("Low", "low") \
    .withColumnRenamed("Open", "open") \
    .withColumnRenamed("Volume", "volume")

# Validações básicas
print("=== Validações de Qualidade ===")
total = df_bronze.count()
print(f"Total de registros: {total:,}")

# Checar nulos em colunas críticas
nulls_date = df_bronze.filter(col("date").isNull()).count()
nulls_ticker = df_bronze.filter(col("ticker").isNull() | (col("ticker") == "")).count()
print(f"Registros com date nulo: {nulls_date}")
print(f"Registros com ticker nulo/vazio: {nulls_ticker}")

# Checar duplicatas (date + ticker)
duplicates = df_bronze.groupBy("date", "ticker").count().filter(col("count") > 1).count()
print(f"Registros duplicados (date + ticker): {duplicates}")

# Salvar com particionação por ticker
table_name = "workspace.bronze.fii_prices"

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("ticker") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"\n✓ Tabela {table_name} criada com partição por ticker!")

In [0]:
%sql
SELECT 
  ticker,
  date,
  open,
  high,
  low,
  close,
  adj_close,
  volume,
  source_file,
  ingestion_timestamp
FROM workspace.bronze.fii_prices
ORDER BY ticker, date DESC

In [0]:
%sql
SELECT COUNT(*) 
FROM workspace.bronze.fii_prices;

In [0]:
from pyspark.sql.functions import regexp_extract, col, current_timestamp

# Caminho base dos arquivos
base_path = "/Workspace/Users/tropehe@outlook.com/projeto-fiis/ingestion/data"

# Ler todos os arquivos *_dividendos.csv
df_dividendos_raw = spark.read.csv(
    f"{base_path}/*_dividendos.csv",
    header=True,
    inferSchema=True
)

print("=== Schema Original dos Dividendos ===")
df_dividendos_raw.printSchema()

print("\n=== Amostra dos Dados Originais ===")
display(df_dividendos_raw.limit(5))

# Adicionar metadados, ticker e padronizar nomes das colunas
df_dividendos_bronze = df_dividendos_raw \
    .withColumn("ticker", regexp_extract(col("_metadata.file_path"), r"([A-Z0-9]+)_dividendos\.csv$", 1)) \
    .withColumn("source_file", col("_metadata.file_path")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumnRenamed("y", "year") \
    .withColumnRenamed("m", "month") \
    .withColumnRenamed("d", "day") \
    .withColumnRenamed("ad", "approval_date") \
    .withColumnRenamed("ed", "ex_dividend_date") \
    .withColumnRenamed("pd", "payment_date") \
    .withColumnRenamed("et", "event_type") \
    .withColumnRenamed("etd", "event_type_description") \
    .withColumnRenamed("v", "value") \
    .withColumnRenamed("ov", "original_value") \
    .withColumnRenamed("sv", "string_value") \
    .withColumnRenamed("sov", "string_original_value") \
    .withColumnRenamed("adj", "adjusted")

# Contar registros
total_dividendos = df_dividendos_bronze.count()
print(f"\n=== Total de Registros: {total_dividendos:,} ===")

# Contar por ticker
print("\n=== Registros por Ticker ===")
df_dividendos_count = df_dividendos_bronze.groupBy("ticker").count().orderBy("ticker")
display(df_dividendos_count)

# Validações básicas
print("\n=== Validações de Qualidade ===")
nulls_ex_date = df_dividendos_bronze.filter(col("ex_dividend_date").isNull()).count()
nulls_ticker = df_dividendos_bronze.filter(col("ticker").isNull() | (col("ticker") == "")).count()
nulls_value = df_dividendos_bronze.filter(col("value").isNull()).count()
print(f"Registros com ex_dividend_date nulo: {nulls_ex_date}")
print(f"Registros com ticker nulo/vazio: {nulls_ticker}")
print(f"Registros com value nulo: {nulls_value}")

# Checar duplicatas (ex_dividend_date + ticker)
duplicates = df_dividendos_bronze.groupBy("ex_dividend_date", "ticker").count().filter(col("count") > 1).count()
print(f"Registros duplicados (ex_dividend_date + ticker): {duplicates}")

# Mostrar schema final padronizado
print("\n=== Schema Final Padronizado ===")
df_dividendos_bronze.printSchema()

# Mostrar amostra final ordenada
print("\n=== Amostra dos Dados Finais (10 registros mais recentes) ===")
display(df_dividendos_bronze.orderBy(col("ticker"), col("ex_dividend_date").desc()).limit(10))

In [0]:
# Verificar se year, month e day contêm apenas zeros
print("=== Análise das colunas year, month, day ===")
distinct_years = df_dividendos_bronze.select("year").distinct().collect()
distinct_months = df_dividendos_bronze.select("month").distinct().collect()
distinct_days = df_dividendos_bronze.select("day").distinct().collect()

print(f"Valores únicos em 'year': {[row.year for row in distinct_years]}")
print(f"Valores únicos em 'month': {[row.month for row in distinct_months]}")
print(f"Valores únicos em 'day': {[row.day for row in distinct_days]}")

# Verificar se string_value e string_original_value agregam informação além de value e original_value
print("\n=== Análise das colunas string_value e string_original_value ===")

# Comparar value com string_value
print("\nAmostra comparando value vs string_value:")
df_comparison = df_dividendos_bronze.select(
    "ticker",
    "ex_dividend_date",
    "value",
    "string_value",
    "original_value",
    "string_original_value"
).limit(10)
display(df_comparison)

# Verificar se há casos onde string_value != str(value)
print("\n=== Verificação de diferenças ===")

# Checar valores não-nulos
non_null_value = df_dividendos_bronze.filter(col("value").isNotNull()).count()
non_null_string_value = df_dividendos_bronze.filter(col("string_value").isNotNull()).count()
non_null_original = df_dividendos_bronze.filter(col("original_value").isNotNull()).count()
non_null_string_original = df_dividendos_bronze.filter(col("string_original_value").isNotNull()).count()

print(f"Registros com 'value' não-nulo: {non_null_value}")
print(f"Registros com 'string_value' não-nulo: {non_null_string_value}")
print(f"Registros com 'original_value' não-nulo: {non_null_original}")
print(f"Registros com 'string_original_value' não-nulo: {non_null_string_original}")

# Verificar valores distintos em string_original_value
print("\nValores distintos em 'string_original_value':")
distinct_sov = df_dividendos_bronze.select("string_original_value").distinct().collect()
print([row.string_original_value for row in distinct_sov])

In [0]:
# Criar versão otimizada removendo colunas redundantes
df_dividends_optimized = df_dividendos_bronze.select(
    "ex_dividend_date",
    "payment_date",
    "event_type",
    "event_type_description",
    "value",
    "original_value",
    "adjusted",
    "ticker",
    "source_file",
    "ingestion_timestamp"
)

print("=== Schema Otimizado (11 colunas) ===")
df_dividends_optimized.printSchema()

# Contar registros antes de salvar
total_records = df_dividends_optimized.count()
print(f"\n=== Total de registros a serem gravados: {total_records:,} ===")

# Validações finais
print("\n=== Validações Finais ===")
nulls_ex_date = df_dividends_optimized.filter(col("ex_dividend_date").isNull()).count()
nulls_ticker = df_dividends_optimized.filter(col("ticker").isNull() | (col("ticker") == "")).count()
nulls_value = df_dividends_optimized.filter(col("value").isNull()).count()
print(f"Registros com ex_dividend_date nulo: {nulls_ex_date}")
print(f"Registros com ticker nulo/vazio: {nulls_ticker}")
print(f"Registros com value nulo: {nulls_value}")

# Checar duplicatas
duplicates = df_dividends_optimized.groupBy("ex_dividend_date", "ticker").count().filter(col("count") > 1).count()
print(f"Registros duplicados (ex_dividend_date + ticker): {duplicates}")

# Salvar tabela com particionamento por ticker
table_name = "workspace.bronze.fii_dividends"

df_dividends_optimized.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("ticker") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"\n✓ Tabela {table_name} criada com sucesso!")
print(f"✓ {total_records:,} registros gravados")
print(f"✓ Particionada por ticker")

# Mostrar amostra dos dados gravados
print("\n=== Amostra dos Dados Gravados (10 registros) ===")
df_sample = spark.sql(f"""
    SELECT 
        ticker,
        ex_dividend_date,
        payment_date,
        event_type,
        value,
        original_value,
        adjusted,
        ingestion_timestamp
    FROM {table_name}
    ORDER BY ticker, ex_dividend_date DESC
    LIMIT 10
""")
display(df_sample)

In [0]:
%sql
SELECT
  ticker,
  ex_dividend_date,
  payment_date,
  event_type,
  event_type_description,
  value,
  original_value,
  adjusted,
  source_file,
  ingestion_timestamp
FROM workspace.bronze.fii_dividends
ORDER BY ticker, ex_dividend_date DESC

In [0]:
from pyspark.sql.functions import col, current_timestamp, to_date

# Caminho do arquivo IFIX completo
ifix_path = "/Workspace/Users/tropehe@outlook.com/projeto-fiis/ingestion/data/IFIX_completo.csv"

# Ler o arquivo CSV
df_ifix_raw = spark.read.csv(
    ifix_path,
    header=True,
    inferSchema=True
)

print("=== Schema Original do IFIX ===")
df_ifix_raw.printSchema()

print("\n=== Amostra dos Dados Originais ===")
display(df_ifix_raw.limit(5))

# Padronizar nomes das colunas e adicionar metadados
df_ifix_bronze = df_ifix_raw \
    .withColumnRenamed("Data", "date_string") \
    .withColumnRenamed("Fechamento", "close_string")

# Converter date_string para date e close_string para double
df_ifix_bronze = df_ifix_bronze \
    .withColumn("date", to_date(col("date_string"), "dd/MM/yyyy")) \
    .withColumn("close", col("close_string").cast("double")) \
    .withColumn("source_file", col("_metadata.file_path")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .select("date", "close", "source_file", "ingestion_timestamp")

print("\n=== Schema Final Padronizado ===")
df_ifix_bronze.printSchema()

# Contar registros
total_records = df_ifix_bronze.count()
print(f"\n=== Total de Registros: {total_records:,} ===")

# Validações de qualidade
print("\n=== Validações de Qualidade ===")
nulls_date = df_ifix_bronze.filter(col("date").isNull()).count()
nulls_close = df_ifix_bronze.filter(col("close").isNull()).count()
print(f"Registros com date nulo: {nulls_date}")
print(f"Registros com close nulo: {nulls_close}")

# Checar duplicatas por date
duplicates = df_ifix_bronze.groupBy("date").count().filter(col("count") > 1).count()
print(f"Datas duplicadas: {duplicates}")

# Estatísticas adicionais
print("\n=== Estatísticas dos Dados ===")
df_stats = df_ifix_bronze.select("date", "close").describe()
display(df_stats)

# Mostrar amostra final
print("\n=== Amostra dos Dados Finais (10 registros mais recentes) ===")
display(df_ifix_bronze.orderBy(col("date").desc()).limit(10))

In [0]:
# Salvar tabela IFIX bronze
table_name = "workspace.bronze.ifix"

df_ifix_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"✓ Tabela {table_name} criada com sucesso!")
print(f"✓ {total_records:,} registros gravados")
print(f"✓ Período: 2011-01-03 a 2026-08-05 (~15.5 anos)")

# Validar com SELECT COUNT(*)
validation_df = spark.sql(f"SELECT COUNT(*) as total FROM {table_name}")
print("\n=== Validação: SELECT COUNT(*) ===")
display(validation_df)

In [0]:
%sql
SELECT
  date,
  close,
  source_file,
  ingestion_timestamp
FROM workspace.bronze.ifix
ORDER BY date DESC
LIMIT 20